# 04 — Experimental Policy-Matched Development Evaluation

Evaluate the saved v2 detector on its matching local-shadow development images. Keep pretrained, historical S10 and new S20 results separate.

**Inputs:** Completed native v2 training and matching frozen development export, or compact saved metrics.

**Outputs:** runs/v2/eval/metrics.json and paired native matching arrays for new inference.

**Prerequisites:** Preparation/training are separate saved prerequisites. This notebook never starts training.

<a id="section-1"></a>

## 1. Configuration

Explicit opt-ins only. This is a post-pilot exploratory extension, not a confirmed policy.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from vehicle_detection.common import require_paths
from vehicle_detection.experiment_data import load_experiment

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
EXPERIMENT = "v2"
require_paths(config=ROOT / "configs/v2.yaml", shared=ROOT / "configs/data.yaml")
config = load_experiment(ROOT, EXPERIMENT)

from vehicle_detection.common import read_json
from vehicle_detection.evaluation import evaluate_experiment

ALLOW_INFERENCE = False
EVAL_SETTINGS = {"workers": 0}
SAVED_METRICS = ROOT / "docs/issue45-s20/v2_metrics.json"
ALLOW_COMPARISON = False

<a id="section-2"></a>

## 2. Policy–Detector Pair

In [ ]:
if ALLOW_INFERENCE or (ROOT / "runs/v2/eval/metrics.json").exists():
    result = evaluate_experiment(
        ROOT, EXPERIMENT, allow_inference=ALLOW_INFERENCE, settings=EVAL_SETTINGS
    )
elif SAVED_METRICS.exists():
    result = read_json(SAVED_METRICS)
    print("Saved S20 development metrics; no inference occurred.")
else:
    result = None
    print("No completed evaluation. Set ALLOW_INFERENCE=True after training completes.")
if result is not None:
    display(pd.DataFrame([result["overall"]]))
    display(pd.DataFrame(result["classes"]))

<a id="section-3"></a>

## 3. Three-Arm Comparison

Set ALLOW_COMPARISON=True only after v2 evaluation completes. This explicitly permits the two prespecified v0/v1 diagnostic replays, source-quality support checks and matched CPU timings. Existing runs stay unchanged; final-test pixels stay sealed.

In [ ]:
from vehicle_detection.s20_report import run as compare_s20

if ALLOW_COMPARISON or (ROOT / "docs/issue45-s20/summary.json").exists():
    comparison = compare_s20(ROOT, allow_inference=ALLOW_COMPARISON)
    display({arm: record["native"]["overall"] for arm, record in comparison["records"].items()})
else:
    print("Three-arm diagnostics have not executed; see the committed fine-tuning plan.")

<a id="section-4"></a>

## 4. Evaluation Summary

In [ ]:
print("Development, single seed; not an independent test or a released policy.")
print("Native detector timing excludes preprocessing and I/O. Final holdout stays sealed.")